In [0]:
#!pip install findspark 

In [0]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
df = spark.sql("select 'spark' as hello ")
df.show()

In [0]:
# sqlContext ~ spark 


In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Data_Wrangling").getOrCreate()

In [0]:
!pwd

In [0]:
# This is the location where the data file resides
file_location = "/Workspace/Users/giomikecs17@gmail.com/SelfLearning/movie_data_part1.csv"
# Type of file, PySpark also can read other formats such as json, parquet, orc
file_type = "csv"
# As the name suggests, it can read the underlying existing schema if exists
infer_schema = "False"
#You can toggle this option to True or False depending on whether you have header in your file or not
first_row_is_header = "True"
# This is the delimiter that is in your data file
delimiter = "|"

In [0]:
# Bringing all the options together to read the csv file
df = spark.read.format(file_type) \
.option("inferSchema", infer_schema) \
.option("header", first_row_is_header) \
.option("sep", delimiter) \
.load(file_location)

In [0]:
display(df.limit(2))

In [0]:
df = spark.sql("select * from dev.spark_db.movies")
display(df.limit(2))

In [0]:
df.printSchema()

In [0]:
df.dtypes

In [0]:
df.columns

In [0]:
df.count()

In [0]:
# Defining a list to subset the required columns
select_columns=['id', 'budget', 'popularity', 'release_date', 'revenue', 'title']
# Subsetting the required columns from the DataFrame
df=df.select(*select_columns)
# The following command displays the data; by default it shows top 20 rows
#display(df.limit(2))
df.show(5)

In [0]:
# All the operations are done in a single statement, as follows:
df.select('id', 'budget', 'popularity', 'release_date', 'revenue','title').show()
# You also have the option of selecting the columns by index instead of selecting the names from the original DataFrame:
df.select(df[2],df[1],df[6],df[9],df[10],df[14]).show()

In [0]:
df.show(25, False)

In [0]:
from pyspark.sql.functions import *
# .isNull() operator, which returns true when popularity is null; null represents no value
# or nothing. In the third condition, we use isnan to identify NaN (not a number).
df.filter(df['popularity'].isNull()|isnan(df['popularity'])).count()

In [0]:
df.filter(df['popularity'].isNull()).count()

In [0]:
#df.filter((df['popularity']=='')|df['popularity'].isNull()|isnan(df['popularity'])).count()

In [0]:

# If you need to calculate all the missing values in the DataFrame, you can use the following command, WITHOUT SCHEMA
# df.select([count(when((col(c)=='') | col(c).isNull() |isnan(c), c)).alias(c) for c in df.columns]).show()

In [0]:
from pyspark.sql.functions import col, isnan, when, count
from pyspark.sql.types import DoubleType, FloatType

exprs = []

for field in df.schema.fields:
    c = field.name
    
    if isinstance(field.dataType, (DoubleType, FloatType)):
        expr = count(
            when(col(c).isNull() | isnan(col(c)), c)
        ).alias(c)
    else:
        expr = count(
            when(col(c).isNull(), c)
        ).alias(c)
    
    exprs.append(expr)

df.select(exprs).show()

In [0]:
df.groupBy(df['title']).count().show()

In [0]:
df.columns

In [0]:
df.groupBy(df['popularity']).count().orderBy('count', ascending=False).show()

In [0]:
df.groupby(df['title']).count().sort(desc("count")).show(10, False)

In [0]:
df.groupby(df['title']).count().sort(desc("count")).count()

In [0]:
df.count()

In [0]:
# df.groupby("popularity").count().show(5)
# df.groupby(["popularity"]).count().show(5)
df.groupby(col("title")).count().sort("count", ascending=False).show(5)

In [0]:
total = df.count()
result = df.groupBy("title").count().withColumn("percentage",  col("count")/100).sort(col("percentage"),ascending=False)
result.show()

In [0]:
result.select("title","percentage").show(5)

In [0]:
df.groupBy("title").count().withColumn("percentage",  col("count")/100).sort(col("percentage").desc()).show(5)


In [0]:
df.groupBy("title").count().filter("`count`>7").show(5)

In [0]:
del result

In [0]:
df.dtypes

In [0]:
df = df.withColumn('budget',df['budget'].cast("float"))
#After Casting
df.dtypes

In [0]:
df